In [2]:
import cv2
import numpy as np
import os

In [6]:
import cv2
import numpy as np
from skimage.filters import threshold_sauvola

IMG_PATH = "/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/ocr_65e2cf1a_1_gray.jpg"
img = cv2.imread(IMG_PATH, cv2.IMREAD_GRAYSCALE)

# ── 방법 1: Sauvola 이진화 (필기/인쇄 텍스트에 강함) ──────
def sauvola_binary(img: np.ndarray, window_size: int = 25) -> np.ndarray:
    thresh = threshold_sauvola(img, window_size=window_size, k=0.2)
    binary = (img > thresh).astype(np.uint8) * 255
    return binary

# ── 방법 2: Otsu + 모폴로지 (배경 깔끔한 경우에 강함) ──────
def otsu_clean_binary(img: np.ndarray) -> np.ndarray:
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # 노이즈 제거
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return cleaned

# ── 방법 3: CLAHE 후 Sauvola (각인처럼 저대비일 때 최적) ──
def clahe_sauvola_binary(img: np.ndarray) -> np.ndarray:
    # 대비 강화 먼저
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(img)

    # 언샤프 마스킹
    blur = cv2.GaussianBlur(enhanced, (0, 0), 3)
    sharpened = cv2.addWeighted(enhanced, 2.0, blur, -1.0, 0)

    # Sauvola 이진화
    thresh = threshold_sauvola(sharpened, window_size=25, k=0.2)
    binary = (sharpened > thresh).astype(np.uint8) * 255

    # 반전 (각인은 음각이라 반전해야 텍스트가 흰색으로)
    inverted = cv2.bitwise_not(binary)

    return inverted

# ── 결과 저장 ──────────────────────────────────────────────
result1 = sauvola_binary(img)
result2 = otsu_clean_binary(img)
result3 = clahe_sauvola_binary(img)

cv2.imwrite("binary_1_sauvola.jpg", result1)
cv2.imwrite("binary_2_otsu.jpg", result2)
cv2.imwrite("binary_3_clahe_sauvola.jpg", result3)

# OCR에 넘길 최종 후보 - 정방향 + 반전 둘다
cv2.imwrite("binary_final.jpg", result3)
cv2.imwrite("binary_final_inv.jpg", cv2.bitwise_not(result3))

print("완료!")

완료!
